# Data cleaning notebook for final project - Mattias Simma

### saving the tidy datasets into separate files, to keep it cleaner

In [1]:
import pandas as pd
from datasets import load_dataset

In [2]:
# creating dataframe and setting index to datetime column
# you need to have downloaded the file first, calling it weather_data.csv
# note that the path to the file may have changed

hourly_weather = pd.read_csv("../proj_data/weather_data.csv", parse_dates=['utc_timestamp'])
hourly_weather.set_index('utc_timestamp', inplace=True)

In [3]:
hourly_weather.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 350640 entries, 1980-01-01 00:00:00+00:00 to 2019-12-31 23:00:00+00:00
Data columns (total 84 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   AT_temperature                   350640 non-null  float64
 1   AT_radiation_direct_horizontal   350640 non-null  float64
 2   AT_radiation_diffuse_horizontal  350640 non-null  float64
 3   BE_temperature                   350640 non-null  float64
 4   BE_radiation_direct_horizontal   350640 non-null  float64
 5   BE_radiation_diffuse_horizontal  350640 non-null  float64
 6   BG_temperature                   350640 non-null  float64
 7   BG_radiation_direct_horizontal   350640 non-null  float64
 8   BG_radiation_diffuse_horizontal  350640 non-null  float64
 9   CH_temperature                   350640 non-null  float64
 10  CH_radiation_direct_horizontal   350640 non-null  float64
 11  CH_radiation_diffus

In [4]:
hourly_weather.head(2)

,AT_temperature,AT_radiation_direct_horizontal,AT_radiation_diffuse_horizontal,BE_temperature,BE_radiation_direct_horizontal,BE_radiation_diffuse_horizontal,BG_temperature,BG_radiation_direct_horizontal,BG_radiation_diffuse_horizontal,CH_temperature,...,RO_radiation_diffuse_horizontal,SE_temperature,SE_radiation_direct_horizontal,SE_radiation_diffuse_horizontal,SI_temperature,SI_radiation_direct_horizontal,SI_radiation_diffuse_horizontal,SK_temperature,SK_radiation_direct_horizontal,SK_radiation_diffuse_horizontal
utc_timestamp,,,,,,,,,,,,,,,,,,,,,
1980-01-01 00:00:00+00:00,-3.640,0.0,0.0,-0.720,0.0,0.0,4.664,0.0,0.0,-6.287,...,0.0,-3.945,0.0,0.0,-3.055,0.0,0.0,-4.648,0.0,0.0
1980-01-01 01:00:00+00:00,-3.803,0.0,0.0,-1.165,0.0,0.0,4.052,0.0,0.0,-6.602,...,0.0,-4.053,0.0,0.0,-3.272,0.0,0.0,-4.554,0.0,0.0


In [5]:
# melting to super long format

df_melted = hourly_weather.reset_index().melt(id_vars='utc_timestamp',
                                               var_name='variable',
                                               value_name='value')

df_melted.head(2)

,utc_timestamp,variable,value
0,1980-01-01 00:00:00+00:00,AT_temperature,-3.640
1,1980-01-01 01:00:00+00:00,AT_temperature,-3.803


In [6]:
# split variable into country and measurement
df_melted[['country', 'measurement']] = df_melted['variable'].str.split('_', n=1, expand=True)
df_melted.head(2)

,utc_timestamp,variable,value,country,measurement
0,1980-01-01 00:00:00+00:00,AT_temperature,-3.640,AT,temperature
1,1980-01-01 01:00:00+00:00,AT_temperature,-3.803,AT,temperature


In [7]:
# drop the original variable column
df_melted = df_melted.drop('variable', axis=1)
df_melted.head(2)

,utc_timestamp,value,country,measurement
0,1980-01-01 00:00:00+00:00,-3.640,AT,temperature
1,1980-01-01 01:00:00+00:00,-3.803,AT,temperature


In [8]:
# pivot measurements back into separate columns
df_tidy = df_melted.pivot_table(index=['utc_timestamp', 'country'],
                                 columns='measurement',
                                 values='value').reset_index()

df_tidy.head(2)

measurement,utc_timestamp,country,radiation_diffuse_horizontal,radiation_direct_horizontal,temperature
0,1980-01-01 00:00:00+00:00,AT,0.0,0.0,-3.64
1,1980-01-01 00:00:00+00:00,BE,0.0,0.0,-0.72


In [9]:
df_tidy.set_index('utc_timestamp', inplace=True)
df_tidy.head(2)

measurement,country,radiation_diffuse_horizontal,radiation_direct_horizontal,temperature
utc_timestamp,,,,
1980-01-01 00:00:00+00:00,AT,0.0,0.0,-3.64
1980-01-01 00:00:00+00:00,BE,0.0,0.0,-0.72


In [10]:
df_tidy.to_csv('../proj_data/weather_data_tidy.csv')

In [11]:
ds = load_dataset("electricsheepeurope/europe-owid-average-precipitation-per-year")
yearly_rain = ds["train"].to_pandas()

#yearly_rain = pd.read_csv("../proj_data/precipitation.csv")

yearly_rain.head()

,country_name,country_iso3,year,Annual precipitation
0,Albania,ALB,1940,1388.52010
1,Albania,ALB,1941,1350.72610
2,Albania,ALB,1942,931.15094
3,Albania,ALB,1943,904.35675
4,Albania,ALB,1944,1224.71520


In [12]:
yearly_rain.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2683 entries, 0 to 2682
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   country_name          2683 non-null   object 
 1   country_iso3          2683 non-null   object 
 2   year                  2683 non-null   int64  
 3   Annual precipitation  2683 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 84.0+ KB


In [13]:
# checking what countries are in this second dataset

yearly_rain['country_name'].unique()

array(['Albania', 'Andorra', 'Austria', 'Belarus', 'Belgium',
       'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Czechia',
       'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece',
       'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Lithuania',
       'Luxembourg', 'Moldova', 'Montenegro', 'Netherlands',
       'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania',
       'Russia', 'Serbia'], dtype=object)

In [14]:
# assigning each country its 2-letter code, as in the first dataset

import pycountry

def to_iso2(name):
    try:
        return pycountry.countries.lookup(name).alpha_2
    except LookupError:
        return None  # returns None if country name isn't found

yearly_rain['country_iso2'] = yearly_rain['country_name'].apply(to_iso2)

In [15]:
# keeping only the countries also present in the first dataset

countries = ['AT', 'BE', 'BG', 'CZ', 'DE', 'DK', 'EE', 'FI', 'FR',
             'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'NL', 'NO',
             'PL', 'PT', 'RO']

yearly_rain = yearly_rain[yearly_rain['country_iso2'].isin(countries)]
yearly_rain['country_iso2'].unique()

array(['AT', 'BE', 'BG', 'HR', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'HU',
       'IE', 'IT', 'LV', 'LT', 'LU', 'NL', 'NO', 'PL', 'PT', 'RO'],
      dtype=object)

In [16]:
yearly_rain.head()

,country_name,country_iso3,year,Annual precipitation,country_iso2
172,Austria,AUT,1940,1022.29980,AT
173,Austria,AUT,1941,1240.03590,AT
174,Austria,AUT,1942,887.67737,AT
175,Austria,AUT,1943,915.02936,AT
176,Austria,AUT,1944,1162.76390,AT


In [17]:
yearly_rain.drop('country_iso3', axis=1)
yearly_rain.drop('country_name', axis=1)

,country_iso3,year,Annual precipitation,country_iso2
172,AUT,1940,1022.29980,AT
173,AUT,1941,1240.03590,AT
174,AUT,1942,887.67737,AT
175,AUT,1943,915.02936,AT
176,AUT,1944,1162.76390,AT
...,...,...,...,...
2575,ROU,2021,820.94820,RO
2576,ROU,2022,661.82680,RO
2577,ROU,2023,790.39636,RO
2578,ROU,2024,684.43650,RO


In [18]:
# keeping only the years present in the first dataset

time_range = list(range(1980, 2020))

yearly_rain = yearly_rain[yearly_rain['year'].isin(time_range)]

In [19]:
yearly_rain.reset_index(inplace=True)
yearly_rain.drop('index', axis=1, inplace=True)

In [20]:
yearly_rain.set_index('year', inplace=True)

yearly_rain = yearly_rain[['country_iso2', 'Annual precipitation']]

yearly_rain.head()

,country_iso2,Annual precipitation
year,,
1980,AT,1280.2411
1981,AT,1274.2933
1982,AT,1158.5198
1983,AT,1101.7253
1984,AT,1088.9570


In [21]:
yearly_rain.info()

<class 'pandas.core.frame.DataFrame'>
Index: 840 entries, 1980 to 2019
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   country_iso2          840 non-null    object 
 1   Annual precipitation  840 non-null    float64
dtypes: float64(1), object(1)
memory usage: 19.7+ KB


In [22]:
yearly_rain.index = pd.to_datetime(yearly_rain.index.astype(str) + '-12-31 23:00:00')
yearly_rain.head()

,country_iso2,Annual precipitation
year,,
1980-12-31 23:00:00,AT,1280.2411
1981-12-31 23:00:00,AT,1274.2933
1982-12-31 23:00:00,AT,1158.5198
1983-12-31 23:00:00,AT,1101.7253
1984-12-31 23:00:00,AT,1088.9570


In [23]:
yearly_rain = yearly_rain.rename(columns={'country_iso2': 'country', 'Annual precipitation': 'annual_precipitation'})
yearly_rain.head()

,country,annual_precipitation
year,,
1980-12-31 23:00:00,AT,1280.2411
1981-12-31 23:00:00,AT,1274.2933
1982-12-31 23:00:00,AT,1158.5198
1983-12-31 23:00:00,AT,1101.7253
1984-12-31 23:00:00,AT,1088.9570


In [24]:
yearly_rain.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 840 entries, 1980-12-31 23:00:00 to 2019-12-31 23:00:00
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   country               840 non-null    object 
 1   annual_precipitation  840 non-null    float64
dtypes: float64(1), object(1)
memory usage: 19.7+ KB


**NOTE**

In the temperature/radiation dataset there are 7 more countries than in the rain dataset (CH, ES, GB, GR, SE, SI, SK), but since in that dataset there are more variables than in the rain dataset, I will keep them, even though I won't be able to analyze their precipitation.

In [25]:
exclude = []  # add whichever needs to stay iso2

def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return code  # keeps the original if not found

def convert_to3(code):
    if code in exclude:
        return code
    return iso2_to_iso3(code)

df_tidy['country'] = df_tidy['country'].apply(convert_to3)
yearly_rain['country'] = yearly_rain['country'].apply(convert_to3)

yearly_rain.to_csv('../proj_data/precipitation_tidy_streamlit.csv')
df_tidy.to_csv('../proj_data/hourly_weather_tidy_streamlit.csv')